# Ore-type mapping and tonnage forecasting: full run

Run the cells in order. Before you start, replace `YOUR-USERNAME/YOUR-REPO` in the first code cell with your GitHub repository.

In [ ]:
REPO = "https://github.com/YOUR-USERNAME/YOUR-REPO.git"

import os
if not os.path.exists("repo"):
    !git clone -q {REPO} repo
os.chdir("repo")
print(os.getcwd())

## 1. Install the package versions used for the paper

The scripts run in a separate Python process, so a runtime restart is not needed.

In [ ]:
!pip install -q -r requirements.txt

## 2. Check the data

The two data files are already in the repository. If they are missing, this cell tries Kaggle and then asks you to upload them.

In [ ]:
import glob, shutil
os.makedirs("data", exist_ok=True)
need = ["ore_type_data.csv", "simulated_tonnage.csv"]

if not all(os.path.exists(f"data/{f}") for f in need):
    try:
        import kagglehub
        path = kagglehub.dataset_download("alfredofardo/ore-type-classification")
        for f in need:
            hits = glob.glob(f"{path}/**/{f}", recursive=True)
            if hits:
                shutil.copy(hits[0], f"data/{f}")
    except Exception as e:
        print("Kaggle download did not work:", e)

missing = [f for f in need if not os.path.exists(f"data/{f}")]
if missing:
    print("Please upload:", missing)
    from google.colab import files
    for name, content in files.upload().items():
        open(f"data/{name}", "wb").write(content)

print(sorted(os.listdir("data")))

## 3. Run the analysis

This produces all tables, `results.json` and Figures 1–2. It takes a few minutes.

In [ ]:
!COLUMN_FIGS=1 python analysis.py

## 4. Draw Figures 3–6

In [ ]:
!python make_figs.py

## 5. Main tables

In [ ]:
import pandas as pd
for name, f in [("Table 3", "T1_classification_models.csv"), ("Table 4", "T2_calibration.csv"),
                ("Table 5", "T3_selective_prediction.csv"), ("Table 6", "T5_monthly_M-cal.csv"),
                ("Table 8", "T_pit_holdout.csv"), ("Table 9", "T6_policies.csv"), ("Table 10", "T7_forecasting.csv")]:
    print("\n" + name)
    display(pd.read_csv(f"out/tab/{f}"))

## 6. Figures

In [ ]:
from IPython.display import Image, display
for f in ["out/fig_col/Fig1_reliability_with_counts.png", "out/fig_col/Fig2_risk_coverage.png",
          "fig_paper/fig3.png", "fig_paper/fig4.png", "fig_paper/fig5.png", "fig_paper/fig6.png"]:
    print(f)
    display(Image(f, width=420))

## 7. Compare with the results reported in the paper

In [ ]:
bad = []
for f in sorted(glob.glob("results/tables/*.csv")):
    name = os.path.basename(f)
    try:
        pd.testing.assert_frame_equal(pd.read_csv(f), pd.read_csv(f"out/tab/{name}"), check_exact=False, rtol=1e-6, check_dtype=False)
    except AssertionError:
        bad.append(name)
print("All tables match the paper." if not bad else f"Tables that differ: {bad}")